# 🧠 The Adam Optimizer: Combining Momentum and RMSProp

Welcome to the hands-on explanation notebook for the **Adam Optimizer**! In this notebook, we will:
1. Explain the math of Adam: tracking the first moment (momentum) and the second moment (RMSProp), and applying bias correction.
2. Implement the complete **Adam Optimizer from scratch** in Python/NumPy.
3. Compare the trajectories of **Vanilla GD**, **Momentum GD**, and **Adam** on our steep 2D ravine cost function:
   $$f(x, y) = 0.5x^2 + 10y^2$$
4. Visualize their paths on a 2D contour map to observe how Adam achieves fast, smooth, and coordinate-adaptive convergence.
5. Plot loss curves to contrast convergence speeds.
6. Explain **AdamW (Decoupled Weight Decay)** and its role in modern deep neural networks like YOLO.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Defining the Ravine Function and Gradients

Our cost function represents a steep valley:
$$f(x, y) = 0.5x^2 + 10y^2$$

Gradients:
$$\frac{\partial f}{\partial x} = x, \quad \frac{\partial f}{\partial y} = 20y$$

In [ ]:
def cost_ravine(x, y):
    return 0.5 * x**2 + 10.0 * y**2

def grad_ravine(x, y):
    return np.array([x, 20.0 * y])

## 2. Implementing Optimizers from Scratch

Let's write the three optimization loops:
1.  **Vanilla GD:** $\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \nabla J(\mathbf{w}_t)$
2.  **Momentum:** Uses accumulated velocity vector to accelerate along the valley.
3.  **Adam:** Combines momentum ($m_t$) and squared gradient average ($v_t$), applying time-step power bias corrections.

In [ ]:
def optimize_vanilla(start_pos, lr=0.15, epochs=50):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_momentum(start_pos, lr=0.15, beta=0.9, epochs=50):
    pos = np.array(start_pos, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        v = beta * v + lr * grad
        pos -= v
        history.append(pos.copy())
    return np.array(history)

def optimize_adam(start_pos, lr=0.15, beta1=0.9, beta2=0.999, eps=1e-8, epochs=50):
    pos = np.array(start_pos, dtype=float)
    m = np.zeros(2)
    v = np.zeros(2)
    history = [pos.copy()]
    
    for t in range(1, epochs + 1):
        grad = grad_ravine(pos[0], pos[1])
        
        # 1. Update first moment
        m = beta1 * m + (1.0 - beta1) * grad
        
        # 2. Update second moment
        v = beta2 * v + (1.0 - beta2) * (grad ** 2)
        
        # 3. Bias correction
        m_hat = m / (1.0 - beta1 ** t)
        v_hat = v / (1.0 - beta2 ** t)
        
        # 4. Parameter update
        pos -= (lr / (np.sqrt(v_hat) + eps)) * m_hat
        history.append(pos.copy())
        
    return np.array(history)

# Run optimizations starting at (8.0, 4.0)
start = [8.0, 4.0]
path_vanilla = optimize_vanilla(start, lr=0.08)
path_momentum = optimize_momentum(start, lr=0.08, beta=0.8)
path_adam = optimize_adam(start, lr=0.25)

## 3. Visualizing Trajectories over the Contour Map

Let's generate the 2D contour grid and plot the paths.

In [ ]:
x = np.linspace(-10, 10, 150)
y = np.linspace(-5, 5, 150)
X, Y = np.meshgrid(x, y)
Z = cost_ravine(X, Y)

plt.figure(figsize=(12, 8))
contours = plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

# Plot paths
plt.plot(path_vanilla[:, 0], path_vanilla[:, 1], color='red', marker='o', alpha=0.8, linewidth=1.5, label='Vanilla GD')
plt.plot(path_momentum[:, 0], path_momentum[:, 1], color='blue', marker='s', alpha=0.8, linewidth=1.5, label='Momentum GD')
plt.plot(path_adam[:, 0], path_adam[:, 1], color='green', marker='x', linewidth=2.5, label='Adam (Rapid, Smooth)')

plt.scatter(0, 0, color='gold', s=150, marker='*', zorder=5, label='Minimum (0,0)')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(-10, 10)
plt.ylim(-5, 5)
plt.title('Comparison of Optimizers: Vanilla vs. Momentum vs. Adam')
plt.legend()
plt.show()

Look at the plot!
-   **Vanilla GD (Red):** Oscillates wildly due to the steep y-axis walls.
-   **Momentum (Blue):** Smooths vertical wiggles and builds speed along the horizontal floor.
-   **Adam (Green):** Converges even faster and more directly! The adaptive coordinate division dampens the vertical $y$-step size immediately, while keeping the horizontal $x$-steps large, taking a nearly perfect diagonal path to the center.

## 4. Comparing Convergence Speeds

Let's plot the cost reduction curves.

In [ ]:
cost_vanilla = [cost_ravine(p[0], p[1]) for p in path_vanilla]
cost_momentum = [cost_ravine(p[0], p[1]) for p in path_momentum]
cost_adam = [cost_ravine(p[0], p[1]) for p in path_adam]

plt.figure(figsize=(10, 5))
plt.plot(cost_vanilla, color='red', label='Vanilla GD')
plt.plot(cost_momentum, color='blue', label='Momentum GD')
plt.plot(cost_adam, color='green', label='Adam')
plt.yscale('log')
plt.xlabel('Steps')
plt.ylabel('Log Cost')
plt.title('Cost Convergence Comparison (Log Scale)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 💡 Connection to YOLO and Deep Learning
*   **AdamW (Decoupled Weight Decay):** During training, YOLO models utilize **AdamW** rather than basic Adam. In standard Adam, the L2 weight decay penalty gradient ($\lambda \mathbf{w}$) is blended directly into the first and second moment moving averages ($m_t$ and $v_t$). This distorts the weight decay update. AdamW solves this by applying weight decay directly to the parameter value:
    $$\mathbf{w}_{t+1} = \mathbf{w}_t - \text{Adam\_update} - \alpha \lambda \mathbf{w}_t$$
    This preserves the proper mathematical scale of weight regularization, leading to much more stable convergence.